# ChromatinCNN Training on Google Colab

This notebook provides a complete training pipeline for Chromatin State Prediction model on Google Colab GPUs.

## Features
- GPU-accelerated training with CUDA
- Data augmentation (reverse complement, jitter, noise)
- Learning rate warmup + cosine annealing
- Checkpoint saving and resuming
- Early stopping and best model tracking
- RC-averaged predictions for inference


## Cell 1: Setup and Dependencies

Install required packages and set up the environment.


In [ ]:
# Install required packages
%pip install -q torch tqdm pandas numpy matplotlib

# Import libraries
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    device = 'cpu'
    print("CUDA not available. Using CPU.")

# Mount Google Drive (optional, for saving checkpoints)
# Uncomment to mount
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/chromatin_model_checkpoints'
CHECKPOINT_DIR = './checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


## Cell 2: Configuration

All hyperparameters - modify as needed for experimentation.


In [ ]:
# Configuration
config = {
    # Data paths - modify these to point to your data files
    'train_sequences': 'data/train_sequences.csv',
    'train_labels': 'data/train_labels.csv',
    'val_sequences': 'data/val_sequences.csv',
    'val_labels': 'data/val_labels.csv',
    'test_sequences': 'data/testsequences.csv',
    
    # Model architecture
    'n_classes': 18,
    'conv1_filters': 128,
    'conv2_filters': 256,
    'bottleneck_filters':512,
    'kernel1': 19,
    'kernel2': 10,
    'dropout_rate': 0.3,
    
    # Training hyperparameters
    'learning_rate': 1e-2,
    'warmup_epochs': 5,
    'num_epochs': 45,
    'batch_size': 512,
    'num_workers': 4,  # Colab typically works well with 2-4 workers
    'early_stopping_patience': 10,
    
    # Regularization
    'use_l1_regularization': True,
    'l1_weight': 1e-5,
    'label_smoothing': 0.08,
    'weight_decay': 1e-4,
    'gradient_clip_max_norm': 1.0,
    
    # Data augmentation
    'rc_augment': True,
    'jitter_prob': 0.35,
    'jitter_min_len': 170,
    'noise_prob': 0.05,
    'sequence_length': 200,
    'cache_data': True,  # Cache one-hot encodings in memory
}

# Print configuration
print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")


## Cell 3: Data Loading Utilities

Functions and classes for loading and augmenting DNA sequence data.


In [ ]:
def reverse_complement(sequence: str) -> str:
    """Compute reverse complement of a DNA sequence."""
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join([complement[base] for base in reversed(sequence)])


def one_hot_encode(sequence: str) -> np.ndarray:
    """
    Convert DNA sequence to one-hot encoded array.
    
    Args:
        sequence: DNA sequence (A, C, G, T)
    
    Returns:
        One-hot encoded array of shape (200, 4)
        Channels: A=0, C=1, G=2, T=3
    """
    base_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    one_hot = np.zeros((200, 4), dtype=np.float32)
    
    for i, base in enumerate(sequence):
        if base in base_to_idx:
            one_hot[i, base_to_idx[base]] = 1.0
        else:
            one_hot[i, :] = 0.25
    
    return one_hot


class ChromatinDataset(Dataset):
    """
    Dataset for chromatin state prediction with biological augmentations.
    
    Supports:
    - Reverse complement augmentation (on-the-fly)
    - Position jittering
    - Noise injection
    """
    
    def __init__(
        self,
        sequences_file: str,
        labels_file: str = None,
        rc_augment: bool = False,
        jitter_prob: float = 0.3,
        jitter_min_len: int = 180,
        noise_prob: float = 0.01,
        sequence_length: int = 200,
        cache_data: bool = True,
    ):
        self.sequences_file = Path(sequences_file)
        self.labels_file = Path(labels_file) if labels_file else None
        self.rc_augment = rc_augment
        self.jitter_prob = jitter_prob
        self.jitter_min_len = jitter_min_len
        self.noise_prob = noise_prob
        self.sequence_length = sequence_length
        self.cache_data = cache_data
        
        # Load sequences
        print(f"Loading sequences from {self.sequences_file}")
        sequences_df = pd.read_csv(self.sequences_file, header=None)
        
        if sequences_df.shape[1] == 1:
            self.sequences = sequences_df[0].values
        else:
            self.sequences = sequences_df.iloc[:, 1].values
        
        print(f"Loaded {len(self.sequences)} sequences")
        
        # Load labels if provided
        self.labels = None
        if self.labels_file is not None:
            print(f"Loading labels from {self.labels_file}")
            labels_df = pd.read_csv(self.labels_file, header=None)
            
            if labels_df.shape[1] == 1:
                self.labels = labels_df[0].values
            else:
                self.labels = labels_df.iloc[:, 1].values
            
            # Convert labels from 1-18 to 0-17 for PyTorch
            self.labels = self.labels - 1
            print(f"Loaded {len(self.labels)} labels (converted from 1-18 to 0-17)")
        
        # Cache one-hot encodings
        self._cached_sequences = None
        if cache_data:
            print("Caching one-hot encodings in memory...")
            self._cached_sequences = [
                one_hot_encode(seq) for seq in self.sequences
            ]
            print("Cache complete")
        
        self._validate_sequences()
    
    def _validate_sequences(self):
        """Validate sequence lengths."""
        for seq in self.sequences[:10]:  # Check first 10
            if len(seq) != self.sequence_length:
                print(f"Warning: Found sequence with length {len(seq)} (expected {self.sequence_length})")
    
    def __len__(self) -> int:
        return len(self.sequences)
    
    def __getitem__(self, idx: int) -> tuple:
        """Get a single item with optional augmentations."""
        # Get one-hot encoding
        if self._cached_sequences is not None:
            sequence = self._cached_sequences[idx].copy()
        else:
            sequence = one_hot_encode(self.sequences[idx])
        
        # Apply reverse complement augmentation
        if self.rc_augment and random.random() < 0.5:
            sequence = sequence[::-1, [3, 2, 1, 0]]  # Reverse and swap channels
        
        # Apply position jittering
        if self.jitter_prob > 0 and random.random() < self.jitter_prob:
            crop_len = random.randint(self.jitter_min_len, self.sequence_length)
            start_pos = random.randint(0, self.sequence_length - crop_len)
            cropped = sequence[start_pos:start_pos+crop_len, :].copy()
            sequence = np.zeros_like(sequence)
            sequence[:cropped.shape[0], :] = cropped
        
        # Apply noise injection
        if self.noise_prob > 0:
            noise_mask = np.random.random((self.sequence_length, 4)) < self.noise_prob
            noise_sequence = np.random.random((self.sequence_length, 4)).astype(np.float32)
            noise_sequence = noise_sequence / noise_sequence.sum(axis=1, keepdims=True)
            sequence[noise_mask] = noise_sequence[noise_mask]
        
        # Convert to tensor
        sequence_tensor = torch.from_numpy(sequence).float()
        
        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return sequence_tensor, label
        else:
            return sequence_tensor


class ChromatinDataModule:
    """Manage train/val/test datasets and dataloaders."""
    
    def __init__(
        self,
        train_sequences: str,
        train_labels: str,
        val_sequences: str,
        val_labels: str,
        test_sequences: str,
        batch_size: int,
        num_workers: int,
        rc_augment: bool = True,
        jitter_prob: float = 0.3,
        noise_prob: float = 0.01,
        sequence_length: int = 200,
        cache_data: bool = True,
    ):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = device == 'cuda'
        
        print("Initializing datasets...")
        
        # Training dataset with augmentations
        self.train_dataset = ChromatinDataset(
            sequences_file=train_sequences,
            labels_file=train_labels,
            rc_augment=rc_augment,
            jitter_prob=jitter_prob,
            jitter_min_len=config['jitter_min_len'],
            noise_prob=noise_prob,
            sequence_length=sequence_length,
            cache_data=cache_data,
        )
        
        # Validation dataset (no augmentations)
        self.val_dataset = ChromatinDataset(
            sequences_file=val_sequences,
            labels_file=val_labels,
            rc_augment=False,
            jitter_prob=0.0,
            noise_prob=0.0,
            sequence_length=sequence_length,
            cache_data=cache_data,
        )
        
        # Test dataset (no labels, no augmentations)
        self.test_dataset = ChromatinDataset(
            sequences_file=test_sequences,
            labels_file=None,
            rc_augment=False,
            jitter_prob=0.0,
            noise_prob=0.0,
            sequence_length=sequence_length,
            cache_data=cache_data,
        )
        
        print("Datasets initialized")
        
        # Initialize dataloaders
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
    
    def get_train_dataloader(self) -> DataLoader:
        if self.train_loader is None:
            self.train_loader = DataLoader(
                self.train_dataset,
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=self.pin_memory,
                drop_last=True,
                persistent_workers=False,
            )
        return self.train_loader
    
    def get_val_dataloader(self) -> DataLoader:
        if self.val_loader is None:
            self.val_loader = DataLoader(
                self.val_dataset,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=self.pin_memory,
                persistent_workers=False,
            )
        return self.val_loader
    
    def get_test_dataloader(self) -> DataLoader:
        if self.test_loader is None:
            self.test_loader = DataLoader(
                self.test_dataset,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=self.pin_memory,
                persistent_workers=False,
            )
        return self.test_loader
    
    def get_dataset_sizes(self) -> dict:
        return {
            'train': len(self.train_dataset),
            'val': len(self.val_dataset),
            'test': len(self.test_dataset),
        }
    
    def cleanup(self):
        """Clean up dataloader worker processes."""
        if self.train_loader is not None:
            del self.train_loader
            self.train_loader = None
        if self.val_loader is not None:
            del self.val_loader
            self.val_loader = None
        if self.test_loader is not None:
            del self.test_loader
            self.test_loader = None

print("Data loading utilities defined successfully.")


## Cell 4: Model Architecture

ChromatinCNN - interpretability-friendly 1D CNN for chromatin state prediction.


In [ ]:
class ChromatinCNN(nn.Module):
    """
    Interpretability-friendly 1D CNN for chromatin state prediction.
    
    Architecture:
    1. Motif Detection Block - interpretable conv layers
    2. Sparse Bottleneck - SAE attachment point
    3. Spatial Aggregation - global pooling
    4. Decision Block - classification
    """
    
    def __init__(
        self,
        n_classes: int = 18,
        conv1_filters: int = 128,
        conv2_filters: int = 256,
        bottleneck_filters: int = 512,
        kernel1: int = 19,
        kernel2: int = 11,
        dropout_rate: float = 0.3,
        use_l1_regularization: bool = True,
    ):
        super(ChromatinCNN, self).__init__()
        
        self.n_classes = n_classes
        self.use_l1 = use_l1_regularization
        
        # Motif Detection Block
        self.conv1 = nn.Conv1d(
            in_channels=4,  # A, C, G, T
            out_channels=conv1_filters,
            kernel_size=kernel1,
            padding='same',
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(conv1_filters)
        
        self.conv2 = nn.Conv1d(
            in_channels=conv1_filters,
            out_channels=conv2_filters,
            kernel_size=kernel2,
            padding='same',
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(conv2_filters)
        
        # Sparse Bottleneck (SAE attachment point)
        self.bottleneck = nn.Conv1d(
            in_channels=conv2_filters,
            out_channels=bottleneck_filters,
            kernel_size=1,  # 1x1 convolution
            bias=False,
        )
        self.bn_bottleneck = nn.BatchNorm1d(bottleneck_filters)
        
        # Spatial Aggregation
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # Decision Block
        dense_input_size = bottleneck_filters * 2  # max + avg pooling
        self.dense1 = nn.Linear(dense_input_size, 512)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.dense2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.classifier = nn.Linear(256, n_classes)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Kaiming initialization for ReLU layers."""
        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def get_l1_penalty(self) -> torch.Tensor:
        """Compute L1 penalty for first conv layer."""
        if not self.use_l1:
            return torch.tensor(0.0, device=self.conv1.weight.device)
        return torch.mean(torch.abs(self.conv1.weight))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass."""
        # Ensure channel-first format (batch, 4, 200)
        if x.shape[1] != 4:
            x = x.transpose(1, 2)  # (batch, 200, 4) -> (batch, 4, 200)
        
        # Motif Detection Block
        x1 = F.relu(self.bn1(self.conv1(x)))  # (batch, 128, 200)
        x2 = F.relu(self.bn2(self.conv2(x1)))  # (batch, 256, 200)
        
        # Sparse Bottleneck
        x_bottleneck = F.relu(self.bn_bottleneck(self.bottleneck(x2)))  # (batch, 512, 200)
        
        # Spatial Aggregation
        x_max = self.global_max_pool(x_bottleneck)  # (batch, 512, 1)
        x_max = x_max.squeeze(-1)  # (batch, 512)
        
        x_avg = self.global_avg_pool(x_bottleneck)  # (batch, 512, 1)
        x_avg = x_avg.squeeze(-1)  # (batch, 512)
        
        x = torch.cat([x_max, x_avg], dim=1)  # (batch, 1024)
        
        # Decision Block
        x = F.relu(self.dense1(x))
        x = self.dropout1(x)
        
        x = F.relu(self.dense2(x))
        x = self.dropout2(x)
        
        logits = self.classifier(x)  # (batch, n_classes)
        
        return logits
    
    def predict_with_rc_consistency(
        self,
        x: torch.Tensor,
        return_probs: bool = False
    ) -> torch.Tensor:
        """Make predictions with reverse complement averaging."""
        # Forward pass
        logits_orig = self(x)
        
        # Compute reverse complement
        x_rc = x.flip(dims=[1])[:, :, [3, 2, 1, 0]]  # Reverse and complement
        logits_rc = self(x_rc)
        
        # Average predictions
        logits_avg = (logits_orig + logits_rc) / 2
        
        if return_probs:
            return F.softmax(logits_avg, dim=1)
        else:
            return torch.argmax(logits_avg, dim=1)

print("Model architecture defined successfully.")


## Cell 5: Training Functions

Training and validation functions with progress tracking and metrics.


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, epoch, l1_weight=None):
    """Train for one epoch."""
    model.train()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
    for sequences, labels in pbar:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward pass
        logits = model(sequences)
        loss = criterion(logits, labels)
        
        # Add L1 regularization
        if l1_weight is not None:
            l1_penalty = model.get_l1_penalty()
            loss = loss + l1_weight * l1_penalty
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config['gradient_clip_max_norm'])
        
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(logits, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(val_loader, desc="Validation")
    with torch.no_grad():
        for sequences, labels in pbar:
            sequences = sequences.to(device)
            labels = labels.to(device)
            
            # Forward pass
            logits = model(sequences)
            loss = criterion(logits, labels)
            
            # Statistics
            total_loss += loss.item()
            _, predicted = torch.max(logits, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy


def save_checkpoint(model, optimizer, scheduler, epoch, val_acc, val_loss, checkpoint_path):
    """Save model checkpoint."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_accuracy': val_acc,
        'best_val_loss': val_loss,
        'config': config,
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")


def load_checkpoint(checkpoint_path, model, optimizer, scheduler, device):
    """Load model from checkpoint."""
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    best_val_acc = checkpoint['best_val_accuracy']
    
    print(f"Resumed from epoch {epoch} (val_acc: {best_val_acc:.4f})")
    
    return epoch, best_val_acc

print("Training functions defined successfully.")


## Cell 6: Load Data

Load datasets and create dataloaders.


In [ ]:
# Create data module
data_module = ChromatinDataModule(
    train_sequences=config['train_sequences'],
    train_labels=config['train_labels'],
    val_sequences=config['val_sequences'],
    val_labels=config['val_labels'],
    test_sequences=config['test_sequences'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    rc_augment=config['rc_augment'],
    jitter_prob=config['jitter_prob'],
    noise_prob=config['noise_prob'],
    sequence_length=config['sequence_length'],
    cache_data=config['cache_data'],
)

# Display dataset sizes
dataset_sizes = data_module.get_dataset_sizes()
print("\nDataset sizes:")
for split, size in dataset_sizes.items():
    print(f"  {split}: {size:,}")

# Get dataloaders
train_loader = data_module.get_train_dataloader()
val_loader = data_module.get_val_dataloader()
test_loader = data_module.get_test_dataloader()

print(f"\nDataLoaders created with batch size {config['batch_size']}")


## Cell 7: Initialize Model and Training

Create model, optimizer, scheduler, and loss function.


In [ ]:
# Create model
model = ChromatinCNN(
    n_classes=config['n_classes'],
    conv1_filters=config['conv1_filters'],
    conv2_filters=config['conv2_filters'],
    bottleneck_filters=config['bottleneck_filters'],
    kernel1=config['kernel1'],
    kernel2=config['kernel2'],
    dropout_rate=config['dropout_rate'],
    use_l1_regularization=config['use_l1_regularization'],
)

model = model.to(device)
print(f"\nModel created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)

# Setup learning rate scheduler with warmup and cosine annealing
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=1e-2,
    end_factor=1.0,
    total_iters=config['warmup_epochs'],
)

main_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config['num_epochs'] - config['warmup_epochs'],
    eta_min=1e-6,
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, main_scheduler],
    milestones=[config['warmup_epochs']],
)

print(f"Optimizer and scheduler configured with LR={config['learning_rate']}, warmup={config['warmup_epochs']} epochs")

# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
print(f"Loss function: CrossEntropy with label_smoothing={config['label_smoothing']}")


## Cell 8: Training Loop

Main training loop with checkpointing and early stopping.


In [ ]:
# Check for existing checkpoint
resume_from = None
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

if os.path.exists(best_checkpoint_path):
    print(f"Found existing checkpoint at {best_checkpoint_path}")
    resume_from = best_checkpoint_path

# Training setup
start_epoch = 1
best_val_accuracy = 0.0
best_val_loss = float('inf')
patience_counter = 0

# Resume from checkpoint if exists
if resume_from is not None:
    start_epoch, best_val_accuracy = load_checkpoint(
        resume_from, model, optimizer, scheduler, device
    )
    start_epoch += 1  # Start from next epoch

# Training history
history = {
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': [],
    'learning_rates': [],
}

print(f"\nStarting training for {config['num_epochs']} epochs from epoch {start_epoch}...")

try:
    for epoch in range(start_epoch, config['num_epochs'] + 1):
        # Train
        l1_weight = config['l1_weight'] if config['use_l1_regularization'] else None
        train_loss, train_accuracy = train_epoch(
            model, train_loader, criterion, optimizer, device, epoch, l1_weight
        )
        
        # Validate
        val_loss, val_accuracy = validate(model, val_loader, criterion, device)
        
        # Update learning rate
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        # Log metrics
        print(f"\nEpoch {epoch} Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_accuracy:.4f}")
        print(f"  Learning Rate: {current_lr:.6f}")
        print("-" * 60)
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['learning_rates'].append(current_lr)
        
        # Save best checkpoint
        if val_accuracy > best_val_accuracy:
            print(f"Validation accuracy improved from {best_val_accuracy:.4f} to {val_accuracy:.4f}")
            best_val_accuracy = val_accuracy
            best_val_loss = val_loss
            patience_counter = 0
            save_checkpoint(
                model, optimizer, scheduler, epoch,
                val_accuracy, val_loss, best_checkpoint_path
            )
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping triggered after {epoch} epochs")
            break
    
    print(f"\nTraining completed. Best validation accuracy: {best_val_accuracy:.4f}")

finally:
    # Cleanup dataloaders
    data_module.cleanup()


## Cell 9: Training Visualization

Plot training curves to analyze performance.


In [ ]:
# Create plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot loss curves
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy curves
axes[1].plot(history['train_accuracy'], label='Train Acc', marker='o')
axes[1].plot(history['val_accuracy'], label='Val Acc', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot learning rate schedule
axes[2].plot(history['learning_rates'], label='Learning Rate', marker='o', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTraining curves saved to {os.path.join(CHECKPOINT_DIR, 'training_curves.png')}")


## Cell 10: Inference on Test Set

Generate predictions for test sequences using the best model.


In [ ]:
# Load best checkpoint for inference
if os.path.exists(best_checkpoint_path):
    print(f"Loading best model from {best_checkpoint_path}")
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"Model loaded. Epoch: {checkpoint['epoch']}, Val Acc: {checkpoint['best_val_accuracy']:.4f}")
else:
    print("No checkpoint found. Please train the model first.")
    raise RuntimeError("No checkpoint available for inference")

# Generate predictions with RC averaging
all_predictions = []

print("\nGenerating predictions on test set with RC averaging...")
model.eval()

with torch.no_grad():
    for sequences in tqdm(test_loader, desc="Inference"):
        sequences = sequences.to(device)
        
        # Forward pass
        logits_orig = model(sequences)
        
        # Compute reverse complement
        sequences_rc = sequences.flip(dims=[1])[:, :, [3, 2, 1, 0]]
        logits_rc = model(sequences_rc)
        
        # Average predictions
        logits_avg = (logits_orig + logits_rc) / 2
        
        # Get predictions
        predictions = torch.argmax(logits_avg, dim=1)
        all_predictions.append(predictions.cpu().numpy())

# Concatenate all predictions
predictions = np.concatenate(all_predictions)

# Convert from 0-17 back to 1-18
predictions = predictions + 1

print(f"\nGenerated {len(predictions)} predictions")
print(f"Prediction distribution:")
unique, counts = np.unique(predictions, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Label {label}: {count} ({100*count/len(predictions):.2f}%)")


## Cell 11: Save Predictions and Summary

Save predictions to CSV and provide download instructions.


In [ ]:
# Save predictions to CSV
predictions_df = pd.DataFrame(predictions)
predictions_path = os.path.join(CHECKPOINT_DIR, 'predictions.csv')
predictions_df.to_csv(predictions_path, index=False, header=False)

print(f"\nPredictions saved to {predictions_path}")

# Save training history
history_path = os.path.join(CHECKPOINT_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to {history_path}")

# Summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Best validation accuracy: {best_val_accuracy:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(history['train_accuracy'])}")
print(f"Final learning rate: {history['learning_rates'][-1]:.6f}")
print(f"\nFiles saved to {CHECKPOINT_DIR}:")
print(f"  - best_model.pt (best checkpoint)")
print(f"  - predictions.csv (test predictions)")
print(f"  - training_curves.png (visualization)")
print(f"  - training_history.json (metrics)")

# Download instructions
print("\n" + "="*60)
print("DOWNLOADING FILES")
print("="*60)

if CHECKPOINT_DIR.startswith('/content/drive'):
    print("Files saved to Google Drive. Access them from your Drive.")
else:
    print("\nTo download files from Colab:")
    print("1. Open file browser on left panel")
    print("2. Navigate to 'checkpoints' folder")
    print("3. Right-click on files and select 'Download'")
    print("\n4. Or run in separate cell: !zip -r checkpoints.zip", CHECKPOINT_DIR)

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("- Experiment with different hyperparameters")
print("- Try different learning rates (e.g., 1e-3, 5e-4, 1e-4)")
print("- Adjust warmup epochs (e.g., 3, 5, 10)")
print("- Tune data augmentation probabilities")
print("- Consider increasing model capacity (more filters/layers) if underfitting")
print("- Use saved checkpoint to resume training if Colab session times out")
print("="*60)

print("\nTraining complete!")
